# 固有表現抽出をエンティティ単位で評価する (seqeval)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_ner/seqeval.ipynb)

NLP クックブック 17-09 ／ 固有表現抽出

系列ラベリングの結果を、トークン単位の正解率ではなく、エンティティ単位の適合率・再現率・F1 で評価したい。

In [1]:
!pip install -q seqeval

In [2]:
from seqeval.metrics import accuracy_score, classification_report, f1_score
from seqeval.scheme import IOB2

y_true = [["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "O", "O"], ["B-LOC", "O", "O", "B-PER", "O"], ["O", "O", "B-ORG", "I-ORG", "I-ORG"]]
y_pred = [["B-PER", "I-PER", "O", "B-ORG", "O", "O", "O"], ["B-LOC", "O", "O", "B-PER", "O"], ["O", "O", "B-LOC", "I-LOC", "I-LOC"]]

print("トークン単位の正解率:", round(accuracy_score(y_true, y_pred), 3))
print("エンティティ単位の F1:", round(f1_score(y_true, y_pred), 3))
print(classification_report(y_true, y_pred, digits=3))

トークン単位の正解率: 0.765
エンティティ単位の F1: 0.6
              precision    recall  f1-score   support

         LOC      0.500     1.000     0.667         1
         ORG      0.000     0.000     0.000         2
         PER      1.000     1.000     1.000         2

   micro avg      0.600     0.600     0.600         5
   macro avg      0.500     0.667     0.556         5
weighted avg      0.500     0.600     0.533         5



In [3]:
# 厳密モード: 不正な系列 (B- なしの I-) をエンティティとして数えない
bad_pred = [["I-PER", "I-PER", "O", "B-ORG", "I-ORG", "O", "O"], ["B-LOC", "O", "O", "B-PER", "O"], ["O", "O", "B-ORG", "I-ORG", "I-ORG"]]
print("default:", f1_score(y_true, bad_pred), " strict:", f1_score(y_true, bad_pred, mode="strict", scheme=IOB2))

default: 1.0  strict: 0.888888888888889


- エンティティ単位では、範囲とラベルの両方が完全に一致したときだけ正解。境界が 1 トークンずれても、偽陽性 1 + 偽陰性 1 になる。
- O タグが大半を占めるので、トークン単位の正解率は高く出すぎて指標にならない。
- 論文で報告されるのは通常 micro 平均の F1。